# Multihead Attention

- siehe ch03 -> 02_bonus_efficient-multihead-attention und wähle einen effzizientere imlementation aus
- eventuell auch Buffer einbauen wie in 03_understanding_buffers (wofür sind buffer gut? : um vergangene wörte rnicht wieder neuberechnen zu müssen, die bereits berechnet wurden)


In [3]:
from importlib.metadata import version
import tiktoken
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
print("torch version:", version("torch"))

torch version: 2.10.0


In [4]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

    return dataloader

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head
        
        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [6]:
# Reduced dataset with 100k rows for testing
cloud_url = "https://syncandshare.lrz.de/dl/fiHE8nDPcb4nww3VCn4QmN/reduced_dataset_100k.csv"
# Uncomment the following line to use the full dataset
# cloud_url = "https://syncandshare.lrz.de/dl/finR5gtyQx3FNL5P2hz6H/full_dataset.csv"

try:
    print("Loading dataset from cloud...")
    df = pd.read_csv(cloud_url)
    print("Dataset loaded successfully!\n")
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")

def format_csv(row):
    title = str(row['title'])
    ingredients = str(row['ingredients']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    directions = str(row['directions']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    return f"Recepie: {title}\nIngredients: {ingredients}\nDirections: {directions}"

raw_text = "".join(df.apply(format_csv, axis=1))
print(f"The entire dataset has been successfully converted into a single string.")
print(f"Total character count: {len(raw_text)}")
print("\nPreview of the first 100 characters:")
print(raw_text[:100])

# TODO -> umstellen auf full raw text?
# For a faster computation we only select the first 100 characters of the text for the next steps
raw_text = raw_text[:100]

tokenizer = tiktoken.get_encoding("cl100k_base")
encoded_text = tokenizer.encode(raw_text)

vocab_size = 100277
output_dim = 256
max_len = 1024
context_length = max_len


token_embedding_layer = nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

max_length = 4
dataloader = create_dataloader(raw_text, batch_size=8, max_length=max_length, stride=max_length)

for batch in dataloader:
    x, y = batch

    token_embeddings = token_embedding_layer(x)
    pos_embeddings = pos_embedding_layer(torch.arange(max_length))

    input_embeddings = token_embeddings + pos_embeddings

    break

Loading dataset from cloud...
Dataset loaded successfully!

The entire dataset has been successfully converted into a single string.
Total character count: 47074254

Preview of the first 100 characters:
Recepie: No-Bake Nut Cookies
Ingredients: 1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/


In [9]:
print(len(dataloader))

1


In [7]:
torch.manual_seed(123)

context_length = max_length
d_in = output_dim
d_out = d_in

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

batch = input_embeddings
context_vecs = mha(batch)

print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([7, 4, 256])


In [8]:
print(batch)

tensor([[[-0.2809, -0.1831, -0.4512,  ..., -0.6920,  2.8570,  1.0962],
         [ 0.0147,  0.5792,  0.8578,  ..., -0.4786, -2.1407,  0.2982],
         [ 1.5778, -2.0272, -2.0823,  ..., -0.8894,  1.4432,  2.7675],
         [ 0.9406, -0.0148,  2.9723,  ..., -0.3071, -1.4653,  1.2594]],

        [[-1.9172, -1.5529, -2.3778,  ...,  1.0066,  2.7298,  0.7934],
         [ 0.9670,  2.9738,  1.6779,  ..., -0.9238,  0.8711,  0.7439],
         [ 1.5497,  0.0055, -0.5312,  ...,  0.4493,  2.3459,  2.1927],
         [ 1.5124,  0.7110,  2.1437,  ..., -0.8705,  0.4177, -0.4755]],

        [[-0.8363, -1.9770, -1.2986,  ...,  0.4349,  0.8219,  0.3116],
         [ 0.0147,  0.5792,  0.8578,  ..., -0.4786, -2.1407,  0.2982],
         [-0.2899, -0.1956, -2.4821,  ...,  0.0107,  2.4469,  0.8758],
         [-0.8677,  1.1575,  2.7025,  ...,  0.5918, -0.9129,  2.6725]],

        ...,

        [[-2.1332, -0.9701, -2.6659,  ..., -0.5507,  2.6808, -1.0916],
         [ 1.6460,  2.5204,  2.4514,  ...,  0.3037, -0.93